# 03 — Reversibility and Stage Before Act

**Chapter 2 teaching lab — §2.3 + Figure 2.2**

> **Teaching outcome:** Encode reversibility, blast radius, and stage-before-act.

Flow: *concept → why it matters → diagram → read the implementation → expected behavior →
red-team round → playground → challenge.*

In [ ]:
# Setup: make harnesslab/ importable from any working directory, then load this lab's controls.
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "harnesslab").is_dir())
sys.path.insert(0, str(ROOT))

from harnesslab import Action, AutonomyLevel, Decision, EXECUTES, Reversibility
from harnesslab.reversibility import CATALOG, StagedArtifact, evaluate, stage_before_act
from harnesslab.viz import decisions, risk_matrix
from harnesslab.playground import reversibility_playground
from harnesslab.arena import NAIVE, SECURE, run_gauntlet
from harnesslab.viz import gauntlet_board, journey, show_source
from IPython.display import display

journey(3)

## Concept
Two properties dominate how much oversight an action needs:

- **Reversibility**: can the effect be undone, and at what cost? (read-only → reversible →
  costly-reversible → irreversible)
- **Blast radius**: how much is affected if it goes wrong?

**Stage before act** turns a risky action into two steps: produce an inspectable artifact (a diff, a
draft, a dry-run plan), then execute only that artifact. Staging converts "trust the agent" into
"inspect the thing the agent will do".

## Why this matters
Reversibility written in a design doc does not stop anything; reversibility encoded in the policy
function does. And the labels must come from the **harness's catalog**, never from the agent's
proposal. An agent that can label its own actions "reversible" has granted itself authority.

## Diagram (Figure 2.2)
```mermaid
flowchart TD
  A[Action looked up in the harness catalog] --> I{irreversible?}
  I -->|yes| AP[REQUIRE_APPROVAL]
  I -->|no| B{"blast radius ≥ 5?"}
  B -->|yes| ST[STAGE]
  B -->|no| C{costly to undo?}
  C -->|yes| MO[MONITOR]
  C -->|no| AL[ALLOW]
  AP & ST --> SA[StagedArtifact] --> R[human or automated review] --> X[execute exactly that artifact]
  classDef approval fill:#7c3aed22,stroke:#7c3aed
  classDef stage fill:#d9770622,stroke:#d97706
  classDef monitor fill:#0891b222,stroke:#0891b2
  classDef allow fill:#16a34a22,stroke:#16a34a
  class AP approval
  class ST stage
  class MO monitor
  class AL allow
```

## Read the implementation

In [ ]:
show_source(evaluate, stage_before_act)

## Experiment: the catalog, and the whole policy at a glance

In [ ]:
display(decisions([(name, evaluate(a), f"{a.reversibility.value}, blast {a.blast_radius}")
                   for name, a in CATALOG.items()], title="Harness catalog"))
risk_matrix(evaluate)

## Expected behavior

In [ ]:
assert {n: evaluate(a) for n, a in CATALOG.items()} == {
    "search": Decision.ALLOW,
    "draft": Decision.ALLOW,
    "temp_resource": Decision.MONITOR,
    "bulk_retag": Decision.STAGE,
    "publish": Decision.REQUIRE_APPROVAL,
}
staged = stage_before_act("publish", {"report_id": "R-104"})
assert isinstance(staged, StagedArtifact) and staged.decision is Decision.REQUIRE_APPROVAL
assert stage_before_act("search", {"q": "Q3"}) is Decision.ALLOW
staged

## Red-team round (§2.3)
The same attacks run against two defense sets: **HarnessLab** (what this lab built) and
**Shortcuts** (the patterns the chapter warns about). Every attack must be blocked by HarnessLab.
The Shortcuts defense evaluates the risk labels the proposal carries.

In [ ]:
secure, shortcuts = run_gauntlet(SECURE, "2.3"), run_gauntlet(NAIVE, "2.3")
assert all(r.blocked for r in secure), [r.attack.name for r in secure if not r.blocked]
gauntlet_board(secure, shortcuts, title="Gauntlet · §2.3")

## Playground
Pick a reversibility and blast radius; your cell is outlined on the matrix.

In [ ]:
reversibility_playground()

## Challenge
Add catalog entries and predict each decision *before* running: a customer email, a production service
restart, a fund transfer, a temporary cloud resource, and a database deletion. For any result that
surprises you, decide whether the policy or your labels are wrong.